# LightGBM sweep agent (M3)

Runs cells of the W&B Sweep `sweeps/lgbm.yaml` in Colab. Each cell scores one LightGBM configuration by leave-one-year-out on both bloom targets and logs the result to `within-noise/bluebonnet-bloom`.

**Runtime:** CPU is enough (no GPU needed; each cell takes about a minute).
**Secret:** add `WANDB_API_KEY` under Colab's key icon (Secrets) and allow this notebook to read it.
**Before running:** on your machine, `uv run scripts/04_upload_dataset.py` (so the dataset artifact exists) and `uv run scripts/07_sweep.py --create` (prints the sweep id to paste below).

## 1. Get the code

In [ ]:
import os, subprocess, sys
REPO, BRANCH = "kvenanzi/bluebonnet", "main"
URL = f"https://github.com/{REPO}.git"
if not os.path.isdir("bluebonnet"):
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, URL], check=True)
else:
    subprocess.run(["git", "-C", "bluebonnet", "checkout", "-q", BRANCH], check=True)
    subprocess.run(["git", "-C", "bluebonnet", "pull", "-q"], check=True)
# Import the package from the clone rather than `pip install -e`, which would need a kernel restart.
sys.path.insert(0, os.path.abspath("bluebonnet"))
for m in [m for m in sys.modules if m.startswith("bluebonnet")]:
    del sys.modules[m]

## 2. Install only what Colab lacks

Colab already has numpy, pandas, scikit-learn, and LightGBM; upgrading them would force a restart.

In [ ]:
%pip install -q wandb pyarrow

## 3. Log in to W&B

In [ ]:
import wandb
from google.colab import userdata
os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
wandb.login()

## 4. Run the agent

The data come from the W&B artifact `bluebonnet-seasons:latest`, the same table the local scripts read from `data/processed/`.

In [ ]:
from bluebonnet.sweep import run_one
from bluebonnet import WANDB_ENTITY, WANDB_PROJECT

SWEEP_ID = "paste-sweep-id-here"
COUNT = None   # None: keep going until the grid is finished
wandb.agent(SWEEP_ID, function=lambda: run_one(from_artifact=True, tags=("sweep", "colab")),
            count=COUNT, entity=WANDB_ENTITY, project=WANDB_PROJECT)

## Afterwards

- In the W&B sweep page, open the parallel-coordinates panel: each line is one configuration, coloured by `logit/rmse`.
- Back on your machine: `uv run scripts/06_hypotheses.py` reads the best cell by `logit/rmse` and reports it against ridge for H4.